# CMS Hospital Readmission Risk Analysis
## Notebook 1 — Data Collection & Generation

**Project:** Healthcare Analytics — 30-Day Readmission Risk Prediction  
**Dataset:** Synthetic CMS-style inpatient data (5,000 patient encounters)  
**Output:** `data/raw/cms_patients_raw.csv` + `data/raw/data_dictionary.json`

---

### What this notebook does
This notebook simulates a realistic CMS (Centers for Medicare & Medicaid Services) hospital inpatient dataset. The data mirrors the structure of real Hospital Inpatient Prospective Payment System (IPPS) records, using clinically grounded statistical distributions for all variables.

**5% of records are intentionally dirtied** to demonstrate the cleansing step in Notebook 2.

### 1.1 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')
print(f'pandas  version: {pd.__version__}')
print(f'numpy   version: {np.__version__}')

### 1.2 — Setup output folders

In [ ]:
# Create folder structure
for folder in ['data/raw', 'data/clean', 'data/engineered', 'data/outputs', 'reports']:
    os.makedirs(folder, exist_ok=True)

print('Folder structure created:')
for folder in ['data/raw', 'data/clean', 'data/engineered', 'data/outputs', 'reports']:
    print(f'  {folder}/')

### 1.3 — Define constants and reference data

In [ ]:
np.random.seed(42)
N = 5000

HOSPITALS = [
    'General Medical Center', "St. Mary's Hospital",
    'University Health System', 'Regional Medical Center',
    'Community Hospital', 'Mercy Medical Center',
]
CONDITIONS = [
    'Heart Failure', 'Pneumonia', 'COPD', 'Hip/Knee Replacement',
    'CABG', 'Acute MI', 'Sepsis', 'Stroke', 'Diabetes', 'Renal Failure',
]
PAYERS       = ['Medicare', 'Medicaid', 'Private Insurance', 'Self-Pay']
DISPOSITIONS = ['Home', 'Home Health', 'Skilled Nursing Facility',
                'Rehab Facility', 'Long-Term Care', 'AMA']
ETHNICITIES  = ['White', 'Black/AA', 'Hispanic', 'Asian', 'Other']
STATES       = ['CA','TX','FL','NY','PA','OH','IL','GA','NC','MI']

print(f'Dataset size: {N:,} patient records')
print(f'Hospitals: {len(HOSPITALS)}')
print(f'Diagnoses: {len(CONDITIONS)}')
print(f'Payers:    {len(PAYERS)}')

### 1.4 — Generate clinical variables

Each variable uses a statistically appropriate distribution:
- **Age** → Normal distribution (mean 68, SD 14), clipped to 18–95
- **LOS** → Gamma distribution (skewed right — most stays are short, a few are very long)
- **Comorbidity index** → Gamma distribution (most patients have low-moderate burden)
- **BMI** → Normal distribution (mean 28.5, representative of a hospitalised US population)

In [ ]:
ages         = np.random.normal(68, 14, N).clip(18, 95).astype(int)
los          = np.random.gamma(3.5, 1.8, N).clip(1, 30).round(1)
prior_admits = np.random.poisson(1.4, N).clip(0, 8)
comorbidity  = np.random.gamma(2.2, 1.1, N).clip(0, 10).round(1)
num_meds     = np.random.poisson(8, N).clip(0, 25)
ed_visits    = np.random.poisson(1.2, N).clip(0, 6)
bmi          = np.random.normal(28.5, 6.5, N).clip(15, 55).round(1)

print('Clinical variable distributions:')
print(f'  Age:              mean={ages.mean():.1f}  min={ages.min()}  max={ages.max()}')
print(f'  LOS (days):       mean={los.mean():.1f}  min={los.min():.1f}  max={los.max():.1f}')
print(f'  Prior admits:     mean={prior_admits.mean():.1f}  min={prior_admits.min()}  max={prior_admits.max()}')
print(f'  Comorbidity:      mean={comorbidity.mean():.1f}  min={comorbidity.min():.1f}  max={comorbidity.max():.1f}')
print(f'  Medications:      mean={num_meds.mean():.1f}  min={num_meds.min()}  max={num_meds.max()}')
print(f'  ED visits (6mo):  mean={ed_visits.mean():.1f}  min={ed_visits.min()}  max={ed_visits.max()}')
print(f'  BMI:              mean={bmi.mean():.1f}  min={bmi.min():.1f}  max={bmi.max():.1f}')

### 1.5 — Generate readmission probability (logistic model)

The 30-day readmission outcome is generated using a logistic function with clinically literature-informed coefficients. This ensures the synthetic data has realistic relationships between predictors and the outcome.

In [ ]:
# Social risk composite score
social_risk = (
    (ages > 75).astype(float) * 0.3
    + (prior_admits > 2).astype(float) * 0.4
    + (comorbidity > 5).astype(float) * 0.3
    + np.random.uniform(0, 0.5, N)
).round(3)

# Logistic readmission probability
logit = (
    -2.8
    + 0.018 * ages
    + 0.12  * los
    + 0.28  * prior_admits
    + 0.20  * comorbidity
    + 0.09  * ed_visits
    + 0.05  * num_meds
    + 0.15  * social_risk
    + np.random.normal(0, 0.3, N)
)
prob_readmit = (1 / (1 + np.exp(-logit))).round(4)
readmitted   = (np.random.uniform(0, 1, N) < prob_readmit).astype(int)

# Cost model
base_cost    = los * np.random.normal(2800, 400, N)
readmit_cost = readmitted * np.random.normal(18500, 4200, N)
total_cost   = (base_cost + readmit_cost).clip(500, 200_000).round(0)

print(f'Readmission rate:    {readmitted.mean():.1%}')
print(f'Avg probability:     {prob_readmit.mean():.3f}')
print(f'Avg total cost:      ${total_cost.mean():,.0f}')
print(f'Avg cost readmitted: ${total_cost[readmitted==1].mean():,.0f}')
print(f'Avg cost not readm.: ${total_cost[readmitted==0].mean():,.0f}')

### 1.6 — Assemble the raw dataframe

In [ ]:
admit_dates     = pd.date_range('2023-01-01', periods=N, freq='2h')[:N]
discharge_dates = admit_dates + pd.to_timedelta(los, unit='D')

df = pd.DataFrame({
    'patient_id':            [f'PT{str(i).zfill(6)}' for i in range(1, N+1)],
    'admit_date':            admit_dates.strftime('%Y-%m-%d'),
    'discharge_date':        discharge_dates.strftime('%Y-%m-%d'),
    'age':                   ages,
    'gender':                np.random.choice(['Male','Female'], N, p=[0.48,0.52]),
    'ethnicity':             np.random.choice(ETHNICITIES, N, p=[0.55,0.18,0.16,0.06,0.05]),
    'hospital':              np.random.choice(HOSPITALS, N),
    'state':                 np.random.choice(STATES, N),
    'primary_diagnosis':     np.random.choice(CONDITIONS, N),
    'payer':                 np.random.choice(PAYERS, N, p=[0.52,0.18,0.24,0.06]),
    'discharge_disposition': np.random.choice(DISPOSITIONS, N, p=[0.40,0.22,0.18,0.10,0.07,0.03]),
    'length_of_stay_days':   los,
    'prior_admissions_12mo': prior_admits,
    'comorbidity_index':     comorbidity,
    'num_medications':       num_meds,
    'ed_visits_prior_6mo':   ed_visits,
    'bmi':                   bmi,
    'social_risk_score':     social_risk,
    'total_cost_usd':        total_cost.astype(int),
    'readmitted_30d':        readmitted,
})

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

### 1.7 — Inject dirty records (5% of data)

To simulate real-world data quality issues, we intentionally introduce errors into 5% of records. These will be detected and corrected in Notebook 2.

In [ ]:
dirty_idx = np.random.choice(N, int(N * 0.05), replace=False)

df.loc[dirty_idx[:25],  'age']                 = -1      # invalid age
df.loc[dirty_idx[25:50],'bmi']                 = np.nan  # missing BMI
df.loc[dirty_idx[50:75],'total_cost_usd']      = 0       # zero cost
df.loc[dirty_idx[75:],  'length_of_stay_days'] = np.nan  # missing LOS

print(f'Dirty records injected: {len(dirty_idx):,} ({len(dirty_idx)/N:.1%} of dataset)')
print(f'  Invalid age (< 18):    25 records')
print(f'  Missing BMI:           25 records')
print(f'  Zero cost:             25 records')
print(f'  Missing LOS:           {len(dirty_idx)-75} records')
print(f'\nNull summary after injection:')
print(df.isnull().sum()[df.isnull().sum() > 0])

### 1.8 — Save raw dataset and data dictionary

In [ ]:
df.to_csv('data/raw/cms_patients_raw.csv', index=False)

data_dict = {
    'patient_id':            {'type': 'string',  'description': 'Unique patient encounter ID', 'example': 'PT000001'},
    'admit_date':            {'type': 'date',    'description': 'Date of hospital admission (YYYY-MM-DD)'},
    'discharge_date':        {'type': 'date',    'description': 'Date of hospital discharge (YYYY-MM-DD)'},
    'age':                   {'type': 'integer', 'description': 'Patient age at admission', 'valid_range': '18–95'},
    'gender':                {'type': 'string',  'description': 'Patient gender', 'allowed': ['Male','Female']},
    'ethnicity':             {'type': 'string',  'description': 'Patient ethnicity', 'allowed': ETHNICITIES},
    'hospital':              {'type': 'string',  'description': 'Treating hospital name', 'allowed': HOSPITALS},
    'state':                 {'type': 'string',  'description': 'US state of hospital'},
    'primary_diagnosis':     {'type': 'string',  'description': 'Primary CMS diagnosis category', 'allowed': CONDITIONS},
    'payer':                 {'type': 'string',  'description': 'Primary insurance payer', 'allowed': PAYERS},
    'discharge_disposition': {'type': 'string',  'description': 'Post-discharge destination', 'allowed': DISPOSITIONS},
    'length_of_stay_days':   {'type': 'float',   'description': 'Hospital length of stay in days', 'valid_range': '1–30'},
    'prior_admissions_12mo': {'type': 'integer', 'description': 'Prior admissions in past 12 months', 'valid_range': '0–8'},
    'comorbidity_index':     {'type': 'float',   'description': 'Charlson Comorbidity Index (adapted)', 'valid_range': '0–10'},
    'num_medications':       {'type': 'integer', 'description': 'Active medications at discharge', 'valid_range': '0–25'},
    'ed_visits_prior_6mo':   {'type': 'integer', 'description': 'ED visits in prior 6 months', 'valid_range': '0–6'},
    'bmi':                   {'type': 'float',   'description': 'Body mass index at admission', 'valid_range': '15–55'},
    'social_risk_score':     {'type': 'float',   'description': 'Composite social vulnerability score', 'valid_range': '0–1.5'},
    'total_cost_usd':        {'type': 'integer', 'description': 'Total encounter cost in USD', 'valid_range': '$500–$200,000'},
    'readmitted_30d':        {'type': 'integer', 'description': '30-day readmission flag (TARGET)', 'allowed': [0, 1]},
}

with open('data/raw/data_dictionary.json', 'w') as f:
    json.dump(data_dict, f, indent=2)

print('Files saved:')
print(f'  data/raw/cms_patients_raw.csv  — {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  data/raw/data_dictionary.json  — {len(data_dict)} column definitions')

### 1.9 — Summary statistics

In [ ]:
print('=== DATASET SUMMARY ===')
print(f'Total records:       {len(df):,}')
print(f'Date range:          {df["admit_date"].min()} to {df["admit_date"].max()}')
print(f'Readmission rate:    {df["readmitted_30d"].mean():.1%}')
print(f'Avg age:             {df["age"].mean():.1f} years')
print(f'Avg LOS:             {df["length_of_stay_days"].mean():.1f} days')
print(f'Avg total cost:      ${df["total_cost_usd"].mean():,.0f}')
print()
print('Payer distribution:')
print(df['payer'].value_counts().to_string())
print()
print('Hospital distribution:')
print(df['hospital'].value_counts().to_string())

---
**Notebook 1 complete.** Output saved to `data/raw/cms_patients_raw.csv`

Next → **Notebook 2: Data Cleansing**